# Day 3 Session 1 — RAG Demos

These demos walk through the core retrieval-augmented generation building blocks you will use as an engineer building McKinsey's consulting knowledge platform. Users (consultants) query the system; your job is to make retrieval fast, accurate, and scalable.

**Demos in this notebook:**
1. Embedding Models — creating embeddings, computing cosine similarity, semantic search
2. Vector Stores — indexing and querying documents with ChromaDB
3. Chunking — section-aware splitting for structured consulting reports

---
## Session Goal

Students will learn the **three pillars of RAG** — embeddings that capture semantic meaning, vector stores that enable fast similarity search, and chunking strategies that preserve document structure. These are the retrieval foundations for the capstone system in Session 3.

By the end of this session you will be able to:
- Create and interpret text embeddings using OpenAI's embedding models
- Index documents in ChromaDB and perform similarity search with metadata filtering
- Choose chunking strategies that preserve the logical structure of consulting reports
- Understand how these pieces compose into a complete RAG pipeline

In [ ]:
!pip install -q langchain langchain-openai langchain-core langchain-text-splitters chromadb openai python-dotenv numpy

---
## Setup

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "false")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "mckinsey-genai-3day")

import openai
import chromadb
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
import numpy as np
import json

print("All imports successful!")

---
## Demo 1: Embedding Models — Creating and Comparing Text Embeddings

Embeddings convert text into numerical vectors in a high-dimensional space where **semantically similar texts are close together**. This is the foundation of all modern retrieval systems.

In a consulting context, embeddings let the platform find relevant strategy insights, industry analyses, and engagement playbooks even when the exact terminology differs — e.g., a user query about "post-merger synergies" should retrieve documents discussing "integration value capture."

### Demo 1a: Create Embeddings for Consulting Texts

We embed five texts — four consulting knowledge snippets and one irrelevant weather sentence — to see how embeddings represent topic relevance.

In [ ]:
# Demo 1a - Create embeddings for consulting texts

client = openai.OpenAI()

texts = [
    "Digital transformation in financial services requires a phased approach starting with core banking modernization.",
    "Post-merger integration success depends on Day-1 readiness and cultural alignment between merging entities.",
    "Omnichannel retail strategy must prioritize seamless customer experience across physical and digital touchpoints.",
    "Supply chain resilience requires diversification of sourcing and near-shoring of critical components.",
    "The weather today is sunny and warm."
]

response = client.embeddings.create(
    model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"),
    input=texts
)

embeddings = [item.embedding for item in response.data]
print(f"Number of embeddings: {len(embeddings)}")
print(f"Embedding dimensions: {len(embeddings[0])}")
print(f"First 5 values of embedding 0: {embeddings[0][:5]}")
print(f"\nEach text is now a {len(embeddings[0])}-dimensional vector.")
print(f"Text 0: '{texts[0][:50]}...'")
print(f"Text 4: '{texts[4]}' (irrelevant control text)")

**Observe:** Each text becomes a 1536-dimensional vector. The 'weather' text has near-zero similarity to all consulting texts — embeddings capture topic relevance. These vectors live in a space where distance corresponds to semantic meaning, not surface-level keyword overlap.

### Demo 1b: Compute and Display Similarity Matrix

Cosine similarity measures how aligned two vectors are: 1.0 = identical direction, 0 = orthogonal (unrelated), -1 = opposite.

In [ ]:
# Demo 1b - Compute cosine similarity matrix

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("Similarity matrix:")
print(f"{'':>45}", end="")
for i in range(len(texts)):
    print(f"  [{i}]", end="")
print()
for i, t1 in enumerate(texts):
    print(f"[{i}] {t1[:40]:>42}", end="")
    for j, t2 in enumerate(texts):
        sim = cosine_similarity(embeddings[i], embeddings[j])
        print(f" {sim:.2f}", end="")
    print()

**Key Insight:** Consulting texts about related topics (digital transformation, retail strategy) show moderate similarity (0.30–0.37). The irrelevant weather text scores near zero. This is how semantic search works — it measures meaning, not keywords.

Notice that:
- Digital transformation [0] and retail strategy [2] share moderate similarity (both discuss business strategy)
- The weather text [4] is essentially orthogonal to all consulting content
- No two different texts score 1.0 — only a text compared with itself

### Demo 1c: Semantic Search Query

Now we embed a user query and rank the corpus by similarity — this is the core of RAG retrieval.

In [ ]:
# Demo 1c - Semantic search with a consulting query

query = "What are best practices for post-merger integration?"
query_emb = client.embeddings.create(
    model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"),
    input=[query]
).data[0].embedding

print(f"Query: {query}")
print(f"Query embedding dimensions: {len(query_emb)}")
print("\nRanked results (by cosine similarity):")
scored = [(cosine_similarity(query_emb, emb), text) for emb, text in zip(embeddings, texts)]
scored.sort(reverse=True)
for score, text in scored:
    print(f"  {score:.4f} | {text}")

**Observe:** The PMI document ranks first (0.62 similarity) even though the query uses "post-merger integration" and the document says "Post-merger integration" — embeddings handle synonyms and rephrasing. The weather text correctly scores negative (completely irrelevant).

This is the fundamental mechanism behind RAG: embed the query, compute similarity against indexed documents, retrieve the top-k most relevant chunks.

---
### QnA Recap — Demo 1

**Q: Why do embeddings work better than keyword search?**
> Embeddings capture semantic meaning. A query about "synergy capture" will match a document about "value creation from integration" even with zero keyword overlap.

**Q: What determines embedding quality?**
> The training data and architecture of the embedding model. OpenAI's `text-embedding-3-small` was trained on diverse internet text, making it a strong general-purpose model.

**Q: What happens if the corpus grows to millions of documents?**
> Computing cosine similarity against every document becomes too slow. That's why we need vector stores (Demo 2) — they use approximate nearest neighbor algorithms (like HNSW) to search in sub-millisecond time.

---
### Try This — Demo 1

**Challenge:** Add a query about a completely different topic (e.g., "What is the best recipe for chocolate cake?") and observe how it scores against the consulting corpus. What does this tell you about using embedding similarity as a relevance threshold?

In [ ]:
# Try This: Add your own off-topic query
# Uncomment and modify:

# off_topic_query = "What is the best recipe for chocolate cake?"
# off_topic_emb = client.embeddings.create(
#     model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"),
#     input=[off_topic_query]
# ).data[0].embedding
# 
# print(f"Query: {off_topic_query}")
# print("Similarities:")
# for text, emb in zip(texts, embeddings):
#     sim = cosine_similarity(off_topic_emb, emb)
#     print(f"  {sim:.4f} | {text[:80]}")

---
## Demo 2: Vector Stores — Indexing and Similarity Search with ChromaDB

A vector store indexes embeddings for fast similarity search. ChromaDB is a lightweight, in-memory vector store perfect for prototyping.

Here we build a **consulting knowledge base** — indexing insights across strategy, operations, M&A, digital transformation, and organizational design so users can quickly surface relevant prior work.

**Key concepts:**
- **Collection**: A named group of vectors (like a table in a database)
- **HNSW index**: The algorithm ChromaDB uses for approximate nearest neighbor search
- **Metadata**: Structured attributes attached to each document for filtering

### Demo 2a: Create ChromaDB Collection and Add Documents

We create a collection with cosine distance metric, embed 8 consulting knowledge documents, and index them with metadata.

In [ ]:
# Demo 2a - Create ChromaDB collection and add documents

chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(
    name="mckinsey_knowledge_base",
    metadata={"hnsw:space": "cosine"}
)

# McKinsey consulting knowledge documents with metadata
documents = [
    "Digital transformation in financial services requires a phased approach: digitize core processes, then build new digital offerings, and finally reimagine the business model.",
    "Post-merger integration should follow a 100-day plan covering Day-1 readiness, synergy capture, cultural integration, and operating model design.",
    "Omnichannel retail strategy must unify inventory, pricing, and customer data across all channels to deliver a seamless experience.",
    "Private equity value creation levers include revenue growth acceleration, operational efficiency, strategic M&A, and balance sheet optimization.",
    "Organizational restructuring requires a clear target operating model, role clarity, and a change management program to drive adoption.",
    "ESG strategy should be embedded into core business operations, not treated as a standalone compliance exercise, to create long-term shareholder value.",
    "Healthcare cost transformation requires addressing clinical variation, supply chain optimization, and administrative simplification simultaneously.",
    "Cross-border M&A transactions require careful due diligence on regulatory, tax, cultural, and operational integration risks."
]

metadatas = [
    {"source": "digital_transformation_playbook", "practice_area": "Digital", "industry": "Financial Services"},
    {"source": "ma_integration_guide", "practice_area": "M&A", "industry": "Cross-Industry"},
    {"source": "retail_strategy_whitepaper", "practice_area": "Retail", "industry": "Retail"},
    {"source": "pe_value_creation_framework", "practice_area": "Private Equity", "industry": "Cross-Industry"},
    {"source": "org_design_handbook", "practice_area": "Organization", "industry": "Cross-Industry"},
    {"source": "esg_strategy_report", "practice_area": "Sustainability", "industry": "Cross-Industry"},
    {"source": "healthcare_cost_study", "practice_area": "Healthcare", "industry": "Healthcare"},
    {"source": "cross_border_ma_guide", "practice_area": "M&A", "industry": "Cross-Industry"},
]

# Embed all documents
client = openai.OpenAI()
emb_response = client.embeddings.create(
    model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"),
    input=documents
)
doc_embeddings = [item.embedding for item in emb_response.data]

# Add to ChromaDB
collection.add(
    documents=documents,
    embeddings=doc_embeddings,
    ids=[f"doc_{i}" for i in range(len(documents))],
    metadatas=metadatas
)
print(f"Indexed {collection.count()} documents in McKinsey knowledge base")
print(f"\nMetadata fields available: practice_area, industry, source")
print(f"Practice areas: {sorted(set(m['practice_area'] for m in metadatas))}")

### Demo 2b: Query the Collection

Now we search the collection with a natural language query. ChromaDB handles the embedding comparison internally via its HNSW index.

In [ ]:
# Demo 2b - Query the ChromaDB collection

query = "How should a retailer approach omnichannel transformation?"
query_emb = client.embeddings.create(
    model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"),
    input=[query]
).data[0].embedding

results = collection.query(
    query_embeddings=[query_emb],
    n_results=3
)

print(f"Query: {query}")
print(f"\nTop 3 results:")
for i, (doc, distance, meta) in enumerate(zip(
    results['documents'][0],
    results['distances'][0],
    results['metadatas'][0]
)):
    print(f"  {i+1}. [dist={distance:.4f}] [{meta['practice_area']} | {meta['industry']}]")
    print(f"     {doc}")
    print()

**Gotcha:** ChromaDB returns `distances`, not similarities. Lower distance = more similar (cosine distance = 1 - cosine similarity). Don't confuse them!

| Metric | Best match | Worst match |
|--------|-----------|-------------|
| Cosine similarity | 1.0 | -1.0 |
| Cosine distance (ChromaDB) | 0.0 | 2.0 |

So a distance of 0.36 means a cosine similarity of 0.64 — quite relevant!

---
### Try This — Demo 2

**Challenge:** Use ChromaDB's `where` parameter to add a metadata filter. Try searching only within the "M&A" practice area. Does the ranking change?

In [ ]:
# Try This: Add a metadata filter for practice_area
# Uncomment and modify:

# filtered_results = collection.query(
#     query_embeddings=[query_emb],
#     n_results=3,
#     where={"practice_area": "M&A"}
# )
# 
# print("Filtered to M&A practice area:")
# for i, (doc, distance, meta) in enumerate(zip(
#     filtered_results['documents'][0],
#     filtered_results['distances'][0],
#     filtered_results['metadatas'][0]
# )):
#     print(f"  {i+1}. [dist={distance:.4f}] {doc[:80]}...")

---
## Demo 3: Section-Aware Chunking for Consulting Reports

How you split documents directly affects retrieval quality. McKinsey documents typically follow a structured format: **Executive Summary, Key Findings, Detailed Analysis, Recommendations, and Appendices**. Section-aware splitting preserves this logical structure by prioritizing section boundaries as split points.

**Why chunking matters:**
- Too large: Chunks contain mixed topics, reducing retrieval precision
- Too small: Chunks lose context, producing incoherent results
- Wrong boundaries: Splitting mid-sentence or mid-section destroys meaning

### Demo 3a: Section-Aware Splitting

We use `RecursiveCharacterTextSplitter` with a separator hierarchy that prioritizes `## ` headers (section boundaries) over arbitrary character positions.

In [ ]:
# Demo 3a - Section-aware splitting (respects consulting report structure)

sample_doc = """# Post-Merger Integration: A Best Practice Guide

Post-merger integration (PMI) is the critical phase that determines whether an M&A transaction delivers its intended value. McKinsey research shows that 70% of mergers fail to achieve projected synergies, most often due to integration execution failures.

## Executive Summary

Successful post-merger integration requires a structured 100-day plan, early synergy identification, cultural alignment, and disciplined execution. Our analysis of 200+ transactions reveals that Day-1 readiness is the single strongest predictor of integration success.

## Key Findings

Three factors distinguish successful integrations from unsuccessful ones:

1. Leadership alignment: Joint leadership teams must be appointed within the first two weeks, with clear decision rights and accountability.
2. Synergy capture: Revenue and cost synergies must be quantified with bottom-up rigor and tracked through a dedicated Integration Management Office (IMO).
3. Cultural integration: Cultural differences must be proactively addressed through joint team workshops, shared values articulation, and visible leadership behavior.

## Recommendations

We recommend a phased approach to integration. Phase 1 (Days 1-30) focuses on stabilization and quick wins. Phase 2 (Days 31-100) addresses operating model design and synergy capture. Phase 3 (Months 4-12) drives full integration and transformation.

## Appendix

Detailed case studies from recent financial services and healthcare M&A transactions are provided in the supplementary materials, including synergy tracking templates and cultural assessment frameworks."""

# Section-aware splitting: prioritizes ## headers as split points
section_splitter = RecursiveCharacterTextSplitter(
    chunk_size=int(os.getenv("CHUNK_SIZE", "300")),
    chunk_overlap=int(os.getenv("CHUNK_OVERLAP", "30")),
    separators=["\n## ", "\n# ", "\n\n", "\n", ". ", " "]
)
section_chunks = section_splitter.split_text(sample_doc)

print(f"Section-aware splitting: {len(section_chunks)} chunks")
print(f"Separator hierarchy: ['\\n## ', '\\n# ', '\\n\\n', '\\n', '. ', ' ']")
print()
for i, chunk in enumerate(section_chunks):
    # Show which section each chunk belongs to
    print(f"  Chunk {i+1} ({len(chunk)} chars): {chunk[:90]}...")

**Key Insight:** Section-aware splitting keeps the 'Executive Summary' section intact as one chunk. Naive splitting might cut it in half, losing coherence.

Notice that the separator hierarchy means:
1. First try to split on `## ` (section headers) — preserves logical sections
2. If a section is still too large, split on `\n\n` (paragraph boundaries)
3. Last resort: split on sentences or words

This means each chunk maps to a meaningful unit of the consulting report.

### Demo 3b: Naive Splitting Comparison

Now we compare with naive fixed-size splitting that ignores document structure.

In [ ]:
# Demo 3b - Naive fixed-size splitting (ignores document structure)

naive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=int(os.getenv("CHUNK_SIZE", "200")),
    chunk_overlap=int(os.getenv("CHUNK_OVERLAP", "30")),
    separators=["\n\n", "\n", ". ", " "]  # No ## header awareness
)
naive_chunks = naive_splitter.split_text(sample_doc)

print(f"Naive fixed-size splitting: {len(naive_chunks)} chunks")
print(f"Separators: ['\\n\\n', '\\n', '. ', ' '] (no ## header awareness)")
print()
for i, chunk in enumerate(naive_chunks):
    print(f"  Chunk {i+1} ({len(chunk)} chars): {chunk[:90]}...")

print(f"\n{'='*60}")
print(f"COMPARISON:")
print(f"  Section-aware: {len(section_chunks)} chunks (split on ## headers first)")
print(f"  Naive:         {len(naive_chunks)} chunks (arbitrary {naive_splitter._chunk_size}-char slices)")
print(f"\nSection-aware preserves document structure by splitting on ## headers first.")
print(f"Naive chunks are arbitrary character-count slices that may split mid-section.")

**Observe:** Both may produce a similar number of chunks here, but section-aware chunks align with logical sections (`## ` headers) while naive chunks are arbitrary 200-char slices. When a consultant asks "What does the executive summary say?", section-aware chunking retrieves the complete summary as a single coherent chunk, while naive chunking might return a fragment that starts mid-sentence.

**Rule of thumb for consulting documents:**
- Use section-aware splitting for structured reports (engagement summaries, strategy decks, research papers)
- Use sentence-level splitting for unstructured text (emails, meeting notes, chat logs)
- Always test retrieval quality with representative queries before deploying

---
### QnA Recap — Demos 2 & 3

**Q: When should I use metadata filtering vs. relying purely on embedding similarity?**
> Use metadata filtering when the user's intent clearly targets a specific practice area, industry, or document type. Pure similarity search is better when the query is exploratory or cross-cutting.

**Q: How do I choose chunk_size?**
> It depends on the granularity of information in your corpus and the context window of your LLM. Typical values: 200-500 chars for fine-grained retrieval, 500-1500 for broader context. Always test with real queries.

**Q: Does chunk_overlap help or hurt?**
> Overlap (typically 10-20% of chunk_size) helps ensure that information at chunk boundaries isn't lost. Too much overlap wastes storage and can return duplicate content.

**Q: Can I combine section-aware chunking with metadata?**
> Absolutely — and you should! Each chunk can inherit metadata from its parent document (practice area, author, date, engagement code). This is what you'll build in the Exercise notebook.

---
## Session Summary

| Pillar | What it does | Key tool | McKinsey use case |
|--------|-------------|----------|-------------------|
| **Embeddings** | Convert text to semantic vectors | `openai.embeddings.create()` | Find relevant prior work regardless of terminology |
| **Vector Stores** | Index and search vectors at scale | ChromaDB `collection.query()` | Sub-millisecond retrieval across thousands of documents |
| **Chunking** | Split documents preserving structure | `RecursiveCharacterTextSplitter` | Keep consulting report sections intact for coherent retrieval |

**Next:** In the Exercise notebook, you will combine all three pillars into a reusable `SearchEngine` class, then extend it with LLM answer generation to build a complete RAG pipeline.